In [2]:
import numpy as np
import random
import time
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter


In [3]:
SEED = 70
np.random.seed(SEED)
random.seed(SEED)

#### Fitness function

##### Fitness of a solution measure by Integrated Brier Score. Implement IBS Score   https://github.com/square/pysurvival/blob/master/pysurvival/utils/metrics.py

In [4]:
def fractional_polynomial(X, degrees):
    Z = np.zeros_like(X, dtype=float)
    for j, d in enumerate(degrees):
        if d == 0:
            Z[:, j] = np.log(X[:, j])
        else:
            Z[:, j] = X[:, j] ** d
    return Z


In [5]:
def linear_predictor(X, beta, degrees):
    return fractional_polynomial(X, degrees) @ beta


In [6]:
def weibull_cumulative_hazard(t, shape, scale):
    return (t / scale) ** shape

In [7]:
def survival_fp(t, X, beta, degrees, shape, scale):
    """
    Returns S(t | x) for all samples
    """
    eta = linear_predictor(X, beta, degrees)
    H0 = weibull_cumulative_hazard(t, shape, scale)
    return np.exp(-H0 * np.exp(eta))

In [8]:
def fit_censoring_km(T, E):
    """
    T: event/censoring times
    E: event indicator (1=event, 0=censored)
    """
    km = KaplanMeierFitter()
    km.fit(T, event_observed=1 - E)
    return km


In [9]:
def brier_score_fp(
    t, X, T, E,
    beta, degrees,
    shape, scale,
    km_censor
):
    S_hat = survival_fp(t, X, beta, degrees, shape, scale)
    Y = (T > t).astype(float)

    G_t = km_censor.predict(t)
    G_T = np.array([km_censor.predict(min(ti, t)) for ti in T])

    weights = np.where(
        (T <= t) & (E == 1),
        1.0 / np.maximum(G_T, 1e-8),
        1.0 / np.maximum(G_t, 1e-8)
    )

    return np.mean(weights * (Y - S_hat) ** 2)

In [10]:
def integrated_brier_score_fp(
    X, T, E,
    beta, degrees,
    shape, scale,
    t_max=None,
    n_times=100
):
    if t_max is None:
        t_max = np.max(T)

    times = np.linspace(0.01, t_max, n_times)
    km_censor = fit_censoring_km(T, E)

    bs = [
        brier_score_fp(
            t, X, T, E,
            beta, degrees,
            shape, scale,
            km_censor
        )
        for t in times
    ]

    return np.trapz(bs, times) / t_max
